In [98]:
def save_checkpoint(model, optimizer, filename = 'checkpoint.pth', epoch = None, val_macro_f1 = None):
    filename = Path(filename)
    filename.parent.mkdir(parents = True, exist_ok = True)
    checkpoint = {'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch, 'val_macro_f1': val_macro_f1}
    torch.save(checkpoint, filename)

In [99]:
def load_checkpoint(filepath, model_class, n_classes):
    filepath = Path(filepath)
    checkpoint = torch.load(filepath, map_location=torch.device('cpu'), weights_only=True)
    model = model_class(n_classes)
    model.load_state_dict(checkpoint['state_dict'])
    return model

In [100]:
def fit_epoch(model, train_loader, criterion, optimizer, device = DEVICE):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    processed_data = 0

    for inputs, labels in tqdm(train_loader, desc='train', leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        preds = torch.argmax(outputs, dim=1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data).item()
        processed_data += inputs.size(0)

    train_loss = running_loss / processed_data
    train_acc = running_corrects / processed_data
    return train_loss, train_acc

In [101]:
def eval_epoch(model, val_loader, criterion, device=DEVICE):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    processed_size = 0
    all_preds = []
    all_labels = []

    for inputs, labels in tqdm(val_loader, desc='valid', leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels).item()
        processed_size += inputs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / processed_size
    val_acc = running_corrects / processed_size
    val_macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return val_loss, val_acc, val_macro_f1

In [102]:
def train(train_dataset, val_dataset, model, epochs, batch_size, optimizer, sampler,
          best_state_save_path = 'best_model.pth', run_name = 'experiment', history = None):

    train_loader = DataLoader(train_dataset, batch_size = batch_size, sampler = sampler, shuffle = False, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, num_workers = 2, pin_memory = True)

    if history is None:
        history = []

    criterion = nn.CrossEntropyLoss()
    best_val_macro_f1 = -67
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            'architecture': model.__class__.__name__,
            'optimizer': optimizer.__class__.__name__,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'weight_decay': optimizer.param_groups[0].get('weight_decay', 0.0),
            'epochs': epochs,
            'batch_size': batch_size,
            'weighted_sampler': sampler,
            'total_params': total_params,
            'trainable_params': trainable_params,
            'n_classes': len(train_dataset.label_encoder.classes_),
            'seed': seed,
            'rescale_size': rescale_size,
            'dataset_source': 'stepanyarullin/interior-design-styles',
            'selection_metric': 'validation_macro_f1',
            'business_task': 'interior_style_tagging_for_real_estate_platform'})

        with tqdm(desc='epoch', total=epochs, leave=True) as pbar_outer:
            for epoch in range(epochs):
                train_loss, train_acc = fit_epoch(model, train_loader, criterion, optimizer)
                val_loss, val_acc, val_macro_f1 = eval_epoch(model, val_loader, criterion)
                history.append((train_loss, train_acc, val_loss, val_acc, val_macro_f1))
                mlflow.log_metrics({'train_loss': train_loss, 'train_accuracy': train_acc, 'val_loss': val_loss, 'val_accuracy': val_acc, 'val_macro_f1': val_macro_f1}, step = epoch + 1)
                pbar_outer.update(1)

                print(f'\nEpoch {(epoch + 1):03d} train_loss: {train_loss:0.4f} val_loss: {val_loss:0.4f} train_acc: {train_acc:0.4f} acc_val: {val_acc:0.4f} macro_f1_val: {val_macro_f1:0.4f}\n')

                if val_macro_f1 > best_val_macro_f1:
                    best_val_macro_f1 = val_macro_f1
                    save_checkpoint(model = model, optimizer = optimizer, filename = best_state_save_path, epoch = epoch + 1, val_macro_f1 = val_macro_f1)
                    print(f'Новая лучшая модель сохранена: val_macro_f1 = {val_macro_f1:.4f} epoch = {epoch + 1}')

        mlflow.log_metric('best_val_macro_f1', best_val_macro_f1)
        history_df = pd.DataFrame(history, columns = ['train_loss', 'train_accuracy', 'val_loss', 'val_accuracy', 'val_macro_f1'])
        history_path = reports_dir / f'{run_name}_train_history.csv'
        history_df.to_csv(history_path, index=False)
        mlflow.log_artifact(str(history_path), artifact_path='history')

        mlflow.log_artifact(str(best_state_save_path), artifact_path = 'checkpoints')
        mlflow.log_artifact(str(label_encoder_path), artifact_path = 'preprocessing')
        mlflow.log_artifact(str(split_manifest_path), artifact_path = 'data_split')

    return history, run_id


Эти три функции выше у нас обучают(функция обучения связана с двумя другими) и потом проверяю на валидационной выборке

In [103]:
def predict(model, test_loader):
  with torch.no_grad():
    model.eval()
    logits = []
    for inputs in test_loader:
      inputs = inputs.to(DEVICE)
      outputs = model(inputs).cpu()
      logits.append(outputs)
  probs = nn.functional.softmax(torch.cat(logits), dim=-1).numpy()
  return probs